# EMAIL SPAM DETECTION PROJECT

In [3]:
import pandas as pd

df = pd.read_csv("SpamAssasin.csv")

# Understanding Dataset

- Numder of email : 5809
- Dimension 5809x3
- ham(0) : 4091
- spam(1) : 1718
- Imbalance Ratio : 2.381257275902212(low)
- Average Email length : 303.175 (long)
- emails that contains html tags : 1099
- emails that contains URL : 1175
- Checked one sample email that contain html tag

In [7]:
df.columns

Index(['subject', 'body', 'label'], dtype='object')

In [37]:
df.isna().sum()

subject         0
body            0
label           0
email_length    0
is_html         0
has_Url         0
dtype: int64

In [28]:
df.head()

#Removing Noise
df = df[['subject','body','label']]

df.isna().sum()

IBD = df['label'].value_counts().max() / df['label'].value_counts().min()
print('Imbalance ratio',IBD)

email_length = []
for text in df['body']:
    word = text.split()
    count = len(word)
    email_length.append(count)

df['email_length']=email_length

df['email_length'].mean()




Imbalance ratio 2.382644146767618


np.float64(303.1751033057851)

In [ ]:
df['is_html'] = df['body'].str.contains(r'html|<div|<br|<p|<a',case=False,na=False)

df['is_html'].value_counts()

is_html
False    4709
True     1099
Name: count, dtype: int64

In [ ]:
df['has_Url'] = df['body'].str.contains(r'http|www',case=False,na=False)
df['has_Url'].value_counts()

has_Url
True     4633
False    1175
Name: count, dtype: int64

In [35]:
df[df['is_html']==True]['body'].iloc[0]

'Hello, have you seen and discussed this article and his approach?\n\nThank you\n\nhttp://www.paulgraham.com/spam.html\n-- "Hell, there are no rules here-- we\'re trying to accomplish something."\n-- Thomas Alva Edison\n\n\n\n\n-------------------------------------------------------\nThis sf.net email is sponsored by: OSDN - Tired of that same old\ncell phone?  Get a new here for FREE!\nhttps://www.inphonic.com/r.asp?r=sourceforge1&refcode1=vs3390\n_______________________________________________\nSpamassassin-devel mailing list\nSpamassassin-devel@lists.sourceforge.net\nhttps://lists.sourceforge.net/lists/listinfo/spamassassin-devel'

# Cleaning Data

In [36]:
df['subject'] = df['subject'].fillna('')

In [38]:
df['email_text'] = df['subject'] + ' ' + df['body']

In [39]:
import re 

def remove_html(text):
    return re.sub(r'<.*?>', ' ',text)

In [40]:

def remove_url(text):
    return re.sub(r'http\S+|www\S', ' ',text)

In [46]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_email(text):
    text = text.lower()
    text = remove_html(text)
    text = remove_url(text)
    text = re.sub(r'[^a-z\s]',' ',text)

    words = text.split()

    cleaned_words=[]
    for word in words:
        if word not in stop_words:
            cleaned_words.append(word)

    return ' '.join(cleaned_words)

In [47]:
df['clean_email'] = df['email_text'].apply(clean_email)

# Feature Engineering

In [50]:
df['url_count'] = df['email_text'].str.count(r'http|www')

In [51]:
df['has_html'] = df['email_text'].str.contains(r'<.*?>', regex =True).astype(int)

In [52]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [53]:
df.columns

Index(['subject', 'body', 'label', 'email_length', 'is_html', 'has_Url',
       'email_text', 'clean_email', 'url_count', 'has_html'],
      dtype='object')

# TRAIN-TEST-SPLIT

In [58]:
X_text = df['clean_email']
y = df['label']

In [59]:
X_train_text,X_test_text,y_train,y_test = train_test_split(X_text,y,test_size=0.2,random_state=42,stratify=y)

In [63]:
from sklearn.linear_model import  LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.naive_bayes import MultinomialNB




# Modeling with piepline and Column Transformer

In [66]:
text_feature = 'clean_email'
numeric_features = ['email_length' , 'url_count','has_html']

preprocessor_nb = ColumnTransformer([
    ('text',CountVectorizer(min_df=2),text_feature),
    ('num','passthrough',numeric_features)
])

nb_pipeline = Pipeline([
    ('preprocess',preprocessor_nb),
    ('model',MultinomialNB(alpha=0.5))
])

nb_pipeline.fit(df.loc[X_train_text.index], y_train)
y_pred_nb = nb_pipeline.predict(df.loc[X_test_text.index])


In [80]:
preprocessor_lr = ColumnTransformer([
    ('text',TfidfVectorizer(min_df=2,max_df=0.9),text_feature),
    ('num','passthrough',numeric_features)
])

lr_pipeline = Pipeline([
    ('preprocess',preprocessor_lr),
    ('model',LogisticRegression(max_iter=1000,class_weight='balanced'))
])

lr_pipeline.fit(df.loc[X_train_text.index], y_train)
y_pred_lr = lr_pipeline.predict(df.loc[X_test_text.index])

# y_prob_lr = lr_pipeline.predict_proba(df.loc[X_test_text.index])[:, 1]


In [68]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix , classification_report


# Comparing Models 

In [69]:
print("Naive Bayes Report")
print(classification_report(y_test,y_pred_nb))

Naive Bayes Report
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       818
           1       0.98      0.90      0.94       344

    accuracy                           0.97      1162
   macro avg       0.97      0.95      0.96      1162
weighted avg       0.97      0.97      0.97      1162



In [70]:
print("Logistic Regression Report")
print(classification_report(y_test,y_pred_lr))

Logistic Regression Report
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       818
           1       0.98      0.92      0.95       344

    accuracy                           0.97      1162
   macro avg       0.97      0.96      0.96      1162
weighted avg       0.97      0.97      0.97      1162



In [71]:
cm_nb = confusion_matrix(y_test,y_pred_nb)
cm_nb

array([[812,   6],
       [ 33, 311]])

In [72]:
cm_lr = confusion_matrix(y_test,y_pred_lr)
cm_lr

array([[810,   8],
       [ 26, 318]])

# Logistic Regression After Tuning :

In [75]:
import numpy as np
threshold = 0.4
y_pred_lr_custom = (y_prob_lr>=threshold).astype(int)



In [77]:
print('Tuned Logistic Regression Report')
print(classification_report(y_test,y_pred_lr_custom))

Tuned Logistic Regression Report
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       818
           1       0.97      0.96      0.96       344

    accuracy                           0.98      1162
   macro avg       0.98      0.97      0.97      1162
weighted avg       0.98      0.98      0.98      1162



- Tuned Class weights = balanced

In [81]:
print("Logistic Regression Report")
print(classification_report(y_test,y_pred_lr))

Logistic Regression Report
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       818
           1       0.97      0.97      0.97       344

    accuracy                           0.98      1162
   macro avg       0.98      0.98      0.98      1162
weighted avg       0.98      0.98      0.98      1162



# Model Testing

In [83]:
new_email = "Congraulation , you won a free ticket , open link to claim"

test_df = pd.DataFrame({
    'clean_email':[clean_email(new_email,)],
    'email_length':[len(new_email.split())],
    'url_count':[0],
    'has_html':[0]
})

prediction = lr_pipeline.predict(test_df)
prediction

array([1])